In [59]:
import torch
import pandas as pd 

In [60]:
data=pd.read_csv(r'C:\Users\acer\Desktop\footballl_qa\football_qa_dataset_full.csv')

In [61]:
data

,question,answer
0,Who won the 1930 FIFA World Cup?,Uruguay
1,Which team was crowned champion of the 1930 FI...,Uruguay
2,Which country hosted the 1930 FIFA World Cup?,Uruguay
3,In which country was the 1930 FIFA World Cup h...,Uruguay
4,Who was the runner-up in the 1930 FIFA World Cup?,Argentina
...,...,...
2877,Which MLS club did Lionel Messi join in 2023?,Inter-Miami
2878,Which club signed Zinedine Zidane from Juventu...,Real-Madrid
2879,Which club signed Luis Figo directly from riva...,Real-Madrid
2880,Which club signed Diego Maradona from Barcelon...,Napoli


In [62]:
#tokenize

def tokenize(text):
    text=text.lower()
    text=text.replace('?','')
    text=text.replace("'","")
    
    return text.split()

In [63]:
tokenize('Who won the 1930 FIFA World Cup?')

['who', 'won', 'the', '1930', 'fifa', 'world', 'cup']

In [64]:
#vocab
vocab={'<UNK>':0}

def build_vocab(row):
    token_question=tokenize(row['question'])
    token_answer=tokenize(row['answer'])
    
    merged_tokens=token_question + token_answer
    
    for token in merged_tokens:
        if token not in vocab:
            vocab[token]=len(vocab)


In [65]:
data.apply(build_vocab,axis=1) #axis=1 means whole row 

0       None
1       None
2       None
3       None
4       None
        ... 
2877    None
2878    None
2879    None
2880    None
2881    None
Length: 2882, dtype: object

In [66]:
vocab

{'<UNK>': 0,
 'who': 1,
 'won': 2,
 'the': 3,
 '1930': 4,
 'fifa': 5,
 'world': 6,
 'cup': 7,
 'uruguay': 8,
 'which': 9,
 'team': 10,
 'was': 11,
 'crowned': 12,
 'champion': 13,
 'of': 14,
 'country': 15,
 'hosted': 16,
 'in': 17,
 'held': 18,
 'runner-up': 19,
 'argentina': 20,
 'lost': 21,
 'final': 22,
 'golden': 23,
 'boot': 24,
 'at': 25,
 'guillermo-stabile': 26,
 'top': 27,
 'scorer': 28,
 'how': 29,
 'many': 30,
 'goals': 31,
 'did': 32,
 'score': 33,
 '8': 34,
 '1934': 35,
 'italy': 36,
 'czechoslovakia': 37,
 'oldrich-nejedly': 38,
 '5': 39,
 '1938': 40,
 'france': 41,
 'hungary': 42,
 'leonidas': 43,
 '7': 44,
 '1950': 45,
 'brazil': 46,
 'ademir': 47,
 '9': 48,
 '1954': 49,
 'westgermany': 50,
 'switzerland': 51,
 'sandor-kocsis': 52,
 '11': 53,
 '1958': 54,
 'sweden': 55,
 'just-fontaine': 56,
 '13': 57,
 '1962': 58,
 'chile': 59,
 '1966': 60,
 'england': 61,
 'eusebio': 62,
 '1970': 63,
 'mexico': 64,
 'gerd-muller': 65,
 '10': 66,
 '1974': 67,
 'netherlands': 68,
 'grz

In [67]:
print(len(vocab))

2040


In [68]:
def text_to_indexs(text,vocab):
    indexed_text=[]
    
    for token in tokenize(text):
        if token in vocab:
            indexed_text.append(vocab[token])
        else:
            indexed_text.append(vocab['<UNK>'])
        
    return indexed_text

In [69]:
text_to_indexs('Which country hosted the 1930 FIFA World Cup?',vocab)

[9, 15, 16, 3, 4, 5, 6, 7]

In [70]:
len(vocab)

2040

In [71]:
from torch.utils.data import Dataset,DataLoader

In [72]:



class QADataset(Dataset):
    def __init__(self,data,vocab):
        self.data=data
        self.vocab=vocab
        
    
    def __len__(self):
        return self.data.shape[0]
    
    def __getitem__(self, index):
        numerical_question=text_to_indexs(self.data.iloc[index]['question'],self.vocab)
        numerical_answer=text_to_indexs(self.data.iloc[index]['answer'],self.vocab)
        
        return torch.tensor(numerical_question),torch.tensor(numerical_answer)

In [73]:
dataset=QADataset(data,vocab)

In [74]:
dataset[2]

(tensor([ 9, 15, 16,  3,  4,  5,  6,  7]), tensor([8]))

In [75]:
dataloader=DataLoader(dataset,batch_size=1,shuffle=True)

In [76]:
for question , answer in dataloader:
    print(question,answer[0])

tensor([[   9,  331,    2,    3, 1497,  334,  368,   17,    3, 1573,  777]]) tensor([357])
tensor([[  9, 331,   2,   3, 164,   7, 332, 333, 334,  17,  91]]) tensor([338])
tensor([[  1,   2,   3, 252, 253,  17, 297]]) tensor([298])
tensor([[  9, 331,   2,   3,  70, 163,   7]]) tensor([351])
tensor([[  1,   2,   3, 166, 437, 436,   7]]) tensor([1073])
tensor([[  9, 331,   2,   3, 305, 163, 999, 334]]) tensor([340])
tensor([[  9,  10,  11,   3,  19,  25, 168, 192]]) tensor([61])
tensor([[  1,   2,   3, 814, 775, 334, 368]]) tensor([360])
tensor([[  9, 331,   2,   3, 180, 164,   7, 131,   7]]) tensor([358])
tensor([[  9, 331,   2,   3, 305, 437, 333, 334]]) tensor([1663])
tensor([[ 29,  30, 333, 334, 194, 105, 363, 364,   2]]) tensor([44])
tensor([[  9, 322, 323,   3, 252, 253, 324]]) tensor([325])
tensor([[   9,  331,    2,    3,  175,  434,    7,   14,  333, 1517]]) tensor([1610])
tensor([[ 148,  111,    3,  918,   14, 1006, 1007]]) tensor([1369])
tensor([[   9,  331,    2,    3,   96,  

tensor([[  9,  15,  16,   3, 236,   5, 326,   6,   7]]) tensor([55])
tensor([[   9,  331,    2,    3, 1480,  334,  368,   17,    3,  794,  777]]) tensor([339])
tensor([[  1,   2,   3, 786, 896, 897, 368]]) tensor([901])
tensor([[  9, 331, 776, 775, 334, 333,  17,   3, 801, 777]]) tensor([357])
tensor([[148,  11,   3, 149,  14,   3, 150,   6,   7, 151, 152, 153,  63]]) tensor([154])
tensor([[   1,   11,    3, 1093, 1002,   14,    3,   10,  402,    2,    3,   78,
            5,    6,    7]]) tensor([272])
tensor([[  9, 331,   2,   3, 186,   5, 331,   6,   7]]) tensor([340])
tensor([[   9,  331,    2,    3, 1965,  334,  368,   17,    3,  864,  777]]) tensor([350])
tensor([[   1,  121,  122,  400,  120,   51, 1193, 1206,   25,    3,   94,    5,
            6,    7]]) tensor([1207])
tensor([[   9,  331,    2,    3, 1497,  334,  368,   17,    3,  856,  777]]) tensor([344])
tensor([[  9,  10,  11,   3,  19,  25, 168, 186]]) tensor([87])
tensor([[  1, 829,   3, 931,  10,  17,   3,  85, 164,   

In [77]:
import torch.nn as nn
vocab_size=len(vocab)

In [78]:
vocab_size

2040

In [79]:
dropout=0.3

In [80]:



class SImpleRNN(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        
        self.embedding=nn.Embedding(vocab_size,embedding_dim=50)
        self.rnn=nn.RNN(50,128,batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc=nn.Linear(128,vocab_size)
        
    def forward(self,question):
        embedded_question=self.embedding(question)
        hidden,final=self.rnn(embedded_question)
        final=self.dropout(final.squeeze(0))
        output=self.fc(final)    
        
        return output

In [81]:
# learning_rate=0.002
epochs=30

In [82]:
model=SImpleRNN(len(vocab))

In [83]:
criterion=nn.CrossEntropyLoss()

optimizer=torch.optim.Adam(model.parameters(),lr=5e-4, weight_decay=1e-5)

In [84]:
#training loop

for epoch in range(epochs):
    total_loss=0
    for question,answer in dataloader:
        optimizer.zero_grad()
        output=model(question)
        loss=criterion(output,answer[0])
        
        loss.backward()
        
        optimizer.step()
        
        total_loss=total_loss+loss.item()
    avg_loss=total_loss/len(dataloader)
    print(f'Epoch {epoch +1},loss:{avg_loss:4f}')

Epoch 1,loss:6.061652
Epoch 2,loss:4.830311
Epoch 3,loss:4.252464
Epoch 4,loss:3.866011
Epoch 5,loss:3.606189
Epoch 6,loss:3.350726
Epoch 7,loss:3.148735
Epoch 8,loss:2.926691
Epoch 9,loss:2.732805
Epoch 10,loss:2.553644
Epoch 11,loss:2.351739
Epoch 12,loss:2.211740
Epoch 13,loss:2.023253
Epoch 14,loss:1.852826
Epoch 15,loss:1.713992
Epoch 16,loss:1.577638
Epoch 17,loss:1.451722
Epoch 18,loss:1.330107
Epoch 19,loss:1.211802
Epoch 20,loss:1.111575
Epoch 21,loss:1.010934
Epoch 22,loss:0.935168
Epoch 23,loss:0.845129
Epoch 24,loss:0.774878
Epoch 25,loss:0.708458
Epoch 26,loss:0.668550
Epoch 27,loss:0.611007
Epoch 28,loss:0.564399
Epoch 29,loss:0.539126
Epoch 30,loss:0.486194


In [166]:
def predict(model,question,threshold=0.5):
    model.eval()
    
    #covert question to numbers
    numerical_question=text_to_indexs(question,vocab)
    
    #tensor
    question_tensor=torch.tensor(numerical_question).unsqueeze(0)
    
    #send to model
    output=model(question_tensor)
    
    #conv=ert logits to proba 
    proba=torch.nn.functional.softmax(output,dim=1)
    
    #find max proba 
    value, index= torch.max(proba,dim=1)
    
    if value < threshold:
        print('idk')
    else:
        
        print(list(vocab.keys())[index])

In [ ]:
predict(model,"team lost the final of 1954 FIFA World Cup?")

idk
